In [1]:
import os
import sys
import numpy as np
import pandas as pd

ROOT = ".." if os.path.exists("../data") else "."
DATA = os.path.join(ROOT, "data")
sys.path.insert(0, os.path.join(ROOT, "scripts", "analysis"))
import study_config as cfg  # single source of truth for cohort/endpoint/feature decisions

def load(name):
    for root, _, files in os.walk(DATA):
        for f in files:
            if f.upper().startswith(name) and f.upper().endswith(".XPT"):
                df = pd.read_sas(os.path.join(root, f))
                num = df.select_dtypes("number").columns
                df[num] = df[num].mask(df[num].abs() < 1e-10, 0)
                print("Loaded", os.path.join(root, f), df.shape)
                return df
    print("NOT FOUND:", name)

demo, cbc, ghb, diq = load("DEMO_L"), load("CBC_L"), load("GHB_L"), load("DIQ_L")

df = (demo.merge(cbc, on="SEQN")
          .merge(ghb.drop(columns="WTPH2YR"), on="SEQN")
          .merge(diq[["SEQN", "DIQ010", "DIQ050", "DIQ070"]], on="SEQN", how="left"))

in_pregnancy_band = df["RIAGENDR"].eq(cfg.PREGNANCY_BAND_SEX) & df["RIDAGEYR"].between(*cfg.PREGNANCY_BAND_AGE)

flow = [("Has CBC and HbA1c records", len(df))]
d = df[df["RIDSTATR"].eq(cfg.REQUIRE_EXAM_STATUS) & df["RIDAGEYR"].ge(cfg.MIN_AGE)]
flow.append((f"Examined (MEC), age {cfg.MIN_AGE}+", len(d)))
d = d[d["DIQ010"].eq(cfg.NO_PRIOR_DIAGNOSIS_CODE)]
flow.append(("No prior diabetes diagnosis (explicit 'no'; borderline/refused/missing excluded)", len(d)))
d = d[~d["DIQ050"].eq(1) & ~d["DIQ070"].eq(1)]
flow.append(("No insulin or diabetic-pill use", len(d)))
d = d[~d["RIDEXPRG"].eq(1) & (~in_pregnancy_band.loc[d.index] | d["RIDEXPRG"].eq(2))]
flow.append(("Pregnancy-clear (confirmed not pregnant for women 20-44)", len(d)))
d = d[(d["WTPH2YR"] > 0) & d["LBXGH"].notna()]
flow.append(("Valid weight and HbA1c", len(d)))
coh = d.copy()

display(pd.DataFrame(flow, columns=["Step", "Participants"]))

rows = []
for label, cut in [("Primary", cfg.PRIMARY_CUT), ("Secondary", cfg.SECONDARY_CUT)]:
    y = (coh["LBXGH"] >= cut).astype(int)
    rows.append({"Outcome": f"{label}: HbA1c >= {cut}%",
                 "Cases (n)": int(y.sum()),
                 "Unweighted %": round(100 * y.mean(), 1),
                 "Weighted %": round(100 * np.average(y, weights=coh["WTPH2YR"]), 1)})
display(pd.DataFrame(rows))

Loaded ../data/raw/2021_2023/DEMO_L.xpt (11933, 27)
Loaded ../data/raw/2021_2023/CBC_L.xpt (8727, 23)
Loaded ../data/raw/2021_2023/GHB_L.xpt (7199, 3)
Loaded ../data/raw/2021_2023/DIQ_L.xpt (11744, 9)


,Step,Participants
0,Has CBC and HbA1c records,7199
1,"Examined (MEC), age 20+",6064
2,No prior diabetes diagnosis (explicit 'no'; bo...,4975
3,No insulin or diabetic-pill use,4934
4,Pregnancy-clear (confirmed not pregnant for wo...,4865
5,Valid weight and HbA1c,4636


,Outcome,Cases (n),Unweighted %,Weighted %
0,Primary: HbA1c >= 6.5%,106,2.3,2.0
1,Secondary: HbA1c >= 5.7%,1356,29.2,25.6


In [2]:
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.metrics import roc_auc_score

bmx, crp, fer = load("BMX_L"), load("HSCRP_L"), load("FERTIN_L")
data = (coh.merge(bmx[["SEQN", "BMXBMI", "BMXWAIST"]], on="SEQN", how="left")
           .merge(crp[["SEQN", "LBXHSCRP"]], on="SEQN", how="left")
           .merge(fer[["SEQN", "LBXFER"]], on="SEQN", how="left"))
data["male"] = (data["RIAGENDR"] == 1).astype(int)
data["log_crp"] = np.log1p(data["LBXHSCRP"])

# The team's CBC feature block (scripts/analysis/study_config.py): no absolute white-cell
# counts, no NRBC, no MCH/MCHC/MPV/basophil % -- 11 features total.
CBC = cfg.CBC_FEATURES
RED = cfg.RED_CELL_FEATURES
NO_RED = [c for c in CBC if c not in RED]
BASE = ["RIDAGEYR", "male", "BMXBMI"]

model = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                      LogisticRegression(max_iter=2000))
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=42)

def evaluate(d, sets, ref, cut=cfg.PRIMARY_CUT):
    d = d.reset_index(drop=True)
    y = (d["LBXGH"] >= cut).astype(int).values
    w = d["WTPH2YR"].values
    splits = list(cv.split(d, y))
    scores = {}
    for name, cols in sets.items():
        s = []
        for tr, te in splits:
            model.fit(d[cols].iloc[tr], y[tr])
            p = model.predict_proba(d[cols].iloc[te])[:, 1]
            s.append(roc_auc_score(y[te], p, sample_weight=w[te]))   # survey-weighted AUROC
        scores[name] = np.array(s)
    print(f"n = {len(d)} | cases (HbA1c >= {cut}%) = {int(y.sum())}")
    return pd.DataFrame([{"Model": n, "Features": len(sets[n]),
                          "Weighted AUROC": round(s.mean(), 3), "SD": round(s.std(), 3),
                          f"Change vs {ref}": round((s - scores[ref]).mean(), 3)}
                         for n, s in scores.items()])
print("Ready")

Loaded ../data/raw/2021_2023/BMX_L.xpt (8860, 22)
Loaded ../data/raw/2021_2023/laboratory/HSCRP_L.xpt (8727, 4)
Loaded ../data/raw/2021_2023/laboratory/FERTIN_L.xpt (2564, 4)
Ready


In [3]:
sets = {
    "A: age + sex": ["RIDAGEYR", "male"],
    "A2: age + sex + BMI": BASE,
    "B: A2 + CBC": BASE + CBC,
    "C: A2 + CBC without red-cell": BASE + NO_RED,
    "D: B + waist": BASE + CBC + ["BMXWAIST"],
    "E: B + hs-CRP": BASE + CBC + ["log_crp"],
}
evaluate(data, sets, ref="A2: age + sex + BMI")

n = 4636 | cases (HbA1c >= 6.5%) = 106


,Model,Features,Weighted AUROC,SD,Change vs A2: age + sex + BMI
0,A: age + sex,2,0.667,0.067,-0.113
1,A2: age + sex + BMI,3,0.779,0.053,0.000
2,B: A2 + CBC,14,0.808,0.038,0.029
3,C: A2 + CBC without red-cell,9,0.798,0.049,0.018
4,D: B + waist,15,0.805,0.041,0.026
5,E: B + hs-CRP,15,0.831,0.039,0.051


In [4]:
hb = data["LBXHGB"]
anemic = ((data["male"] == 1) & (hb < 13)) | ((data["male"] == 0) & (hb < 12))
na = data[hb.notna() & ~anemic]
print("Excluded:", int(anemic.sum()), "anemic,", int(hb.isna().sum()), "missing hemoglobin")
display(evaluate(na, {"A2: age + sex + BMI": BASE, "B: A2 + CBC": BASE + CBC},
                 ref="A2: age + sex + BMI"))

sub = data[data["LBXFER"].notna()]
print(f"Iron subgroup: {len(sub)} people | cases at >= {cfg.PRIMARY_CUT}%:",
      int((sub["LBXGH"] >= cfg.PRIMARY_CUT).sum()))

Excluded: 421 anemic, 17 missing hemoglobin


n = 4198 | cases (HbA1c >= 6.5%) = 95


,Model,Features,Weighted AUROC,SD,Change vs A2: age + sex + BMI
0,A2: age + sex + BMI,3,0.793,0.028,0.000
1,B: A2 + CBC,14,0.812,0.039,0.019


Iron subgroup: 1076 people | cases at >= 6.5%: 15


In [5]:
evaluate(data, sets, ref="A2: age + sex + BMI", cut=cfg.SECONDARY_CUT)

n = 4636 | cases (HbA1c >= 5.7%) = 1356


,Model,Features,Weighted AUROC,SD,Change vs A2: age + sex + BMI
0,A: age + sex,2,0.699,0.020,-0.031
1,A2: age + sex + BMI,3,0.730,0.019,0.000
2,B: A2 + CBC,14,0.783,0.018,0.053
3,C: A2 + CBC without red-cell,9,0.747,0.019,0.016
4,D: B + waist,15,0.782,0.018,0.052
5,E: B + hs-CRP,15,0.789,0.018,0.059
